Check if behavior matches personality over time - porównać zachowanie na początku i na końcu, statystyki, struktura, może sentyment, ewentualnie emocje, różnice między personami - Do simulated agents behave consistently with assigned personalities?

Let's try to do analysis in 14th, 30th, and 60th day of simulation.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.exception import NetworkXError
from scipy.stats import chi2_contingency
import networkx as nx

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import igraph as ig
import leidenalg as la
from sqlalchemy.dialects.mssql.information_schema import columns

from social_graph.pipeline import *
from social_graph.metrics import *
from social_graph.describe import compare_persona_across_simulations

In [2]:
conn3 = sqlite3.connect("data/experiment_set_1/database_server_3.db")
conn4 = sqlite3.connect("data/experiment_set_1/database_server_4.db")
conn5 = sqlite3.connect("data/experiment_set_1/database_server_5.db")

connb1 = sqlite3.connect("data/experiment_set_1/database_server_b1.db")
connb2 = sqlite3.connect("data/experiment_set_1/database_server_b2.db")
connb3 = sqlite3.connect("data/experiment_set_1/database_server_b3.db")

In [3]:
persona_df = pd.read_csv('data/experiment_set_1/persona_df.csv')
persona_df.head()

,Unnamed: 0,id,openness,conscientiousness,extroversion,agreeableness,neuroticism,age,profession,gender,leaning,education_level,persona
0,1,2,consistent/cautious,efficient/organized,solitary/reserved,friendly/compassionate,resilient/confident,young,Skilled_Trades,female,democrat,master,Persona_2
1,2,3,inventive/curious,efficient/organized,outgoing/energetic,friendly/compassionate,sensitive/nervous,old,Skilled_Trades,male,democrat,master,Persona_3
2,3,4,inventive/curious,extravagant/careless,outgoing/energetic,critical/judgmental,sensitive/nervous,young,Science_Academia,female,democrat,master,Persona_4
3,4,5,consistent/cautious,extravagant/careless,solitary/reserved,critical/judgmental,resilient/confident,old,Security,male,democrat,master,Persona_2
4,5,6,consistent/cautious,extravagant/careless,solitary/reserved,friendly/compassionate,sensitive/nervous,old,Science_Academia,male,democrat,master,Persona_3


In [4]:
personas = persona_df[['id', 'persona']].rename(columns={'id': 'follower_id'})
personas.head()

,follower_id,persona
0,2,Persona_2
1,3,Persona_3
2,4,Persona_4
3,5,Persona_2
4,6,Persona_3


## Behaviour stability - Everything the persona has done up to now

### Follow

In [5]:
follow = pd.read_sql("SELECT * FROM follow", conn3)
follow = follow.merge(personas, on='follower_id', how='left')
follow.head()

,user_id,follower_id,id,action,round,persona
0,496,245,1,follow,1,Persona_4
1,743,400,2,follow,1,Persona_3
2,679,743,3,follow,1,Persona_4
3,130,500,4,follow,2,Persona_2
4,825,837,5,follow,2,Persona_1


In [6]:
persona_dict = personas.set_index('follower_id')['persona'].to_dict()

days = [14, 30, 60]
global_summary = []
summaries = []

for day in days:
    follow_t = follow[follow['round'] <= day * 24]

    global_metrics, summary = build_graph_and_analyse(follow_t, persona_dict)
    global_metrics['day'] = day
    summary['day'] = day

    global_summary.append(global_metrics)
    summaries.append(summary)

Graph creation ...

Number of nodes: 628
Number of edges: 2364
Number of connective components: 27
Components sizes: [594, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 1]
Number of nodes (LCC): 594
Number of edges (LCC): 2356

Global metrics ...

Mean degree: 7.53
Density: 0.0060
Diameter: 11
Avg. shortest path: 3.593
Modularity score: 0.335
Persona assortativity: -0.01259069630105292

Local metrics ...


Statistical check (metrics vs. persona) ...

in_degree statistics: p = 0.6615
out_degree statistics: p = 0.6048
total_degree statistics: p = 0.7737
betweenness statistics: p = 0.6236
eigenvector statistics: p = 0.6549
pagerank statistics: p = 0.4462
kcore statistics: p = 0.7451
Graph creation ...

Number of nodes: 776
Number of edges: 4659
Number of connective components: 19
Components sizes: [755, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2, 1]
Number of nodes (LCC): 755
Number of edges (LCC): 4656

Global metrics ...

Mean degree: 12.01
Density: 0

In [7]:
summaries_df = pd.concat(summaries)
summaries_df

,n_nodes,mean_in_degree,mean_out_degree,mean_total_degree,mean_betweenness,median_betweenness,mean_eigenvector,mean_pagerank,mean_kcore,pagerank_ratio,day
persona,,,,,,,,,,,
Persona_1,149,4.402685,4.046980,8.449664,0.002845,0.000444,0.022738,0.001823,4.214765,1.083042,14
Persona_2,153,3.771242,3.843137,7.614379,0.002849,0.000071,0.020531,0.001646,3.810458,0.977475,14
Persona_3,151,4.013245,4.165563,8.178808,0.003216,0.000108,0.021697,0.001705,3.947020,1.013013,14
Persona_4,141,3.666667,3.801418,7.468085,0.002110,0.000127,0.019933,0.001553,3.921986,0.922752,14
Persona_1,179,7.245810,6.748603,13.994413,0.002602,0.000391,0.021380,0.001529,6.754190,1.154721,30
Persona_2,191,5.811518,6.068063,11.879581,0.002222,0.000206,0.017949,0.001269,5.942408,0.958290,30
Persona_3,196,6.433673,6.214286,12.647959,0.002629,0.000154,0.018836,0.001371,5.938776,1.034855,30
Persona_4,189,5.227513,5.666667,10.894180,0.001848,0.000120,0.016582,0.001138,5.619048,0.859471,30
Persona_1,208,11.379808,10.644231,22.024038,0.002292,0.000404,0.020846,0.001338,10.341346,1.173478,60


In [8]:
global_summary_df = pd.concat(global_summary)
global_summary_df

,Simulation,day
Metric,,
Mean degree,7.528662,14
Density,0.006004,14
Diameter,11.000000,14
Avg. shortest path,3.593149,14
Modularity,0.335082,14
Persona assortativity,-0.012591,14
Mean degree,12.007732,30
Density,0.007747,30
Diameter,8.000000,30


In [9]:
rank_df = summaries_df.copy()

metrics = ['mean_in_degree', 'mean_out_degree', 'mean_total_degree', 'mean_betweenness', 'mean_eigenvector', 'mean_pagerank', 'mean_kcore']

for col in metrics:
    rank_df[col + '_rank'] = rank_df.groupby('day')[col] \
                                   .rank(ascending=False, method='min')

rank_df

,n_nodes,mean_in_degree,mean_out_degree,mean_total_degree,mean_betweenness,median_betweenness,mean_eigenvector,mean_pagerank,mean_kcore,pagerank_ratio,day,mean_in_degree_rank,mean_out_degree_rank,mean_total_degree_rank,mean_betweenness_rank,mean_eigenvector_rank,mean_pagerank_rank,mean_kcore_rank
persona,,,,,,,,,,,,,,,,,,
Persona_1,149,4.402685,4.046980,8.449664,0.002845,0.000444,0.022738,0.001823,4.214765,1.083042,14,1.0,2.0,1.0,3.0,1.0,1.0,1.0
Persona_2,153,3.771242,3.843137,7.614379,0.002849,0.000071,0.020531,0.001646,3.810458,0.977475,14,3.0,3.0,3.0,2.0,3.0,3.0,4.0
Persona_3,151,4.013245,4.165563,8.178808,0.003216,0.000108,0.021697,0.001705,3.947020,1.013013,14,2.0,1.0,2.0,1.0,2.0,2.0,2.0
Persona_4,141,3.666667,3.801418,7.468085,0.002110,0.000127,0.019933,0.001553,3.921986,0.922752,14,4.0,4.0,4.0,4.0,4.0,4.0,3.0
Persona_1,179,7.245810,6.748603,13.994413,0.002602,0.000391,0.021380,0.001529,6.754190,1.154721,30,1.0,1.0,1.0,2.0,1.0,1.0,1.0
Persona_2,191,5.811518,6.068063,11.879581,0.002222,0.000206,0.017949,0.001269,5.942408,0.958290,30,3.0,3.0,3.0,3.0,3.0,3.0,2.0
Persona_3,196,6.433673,6.214286,12.647959,0.002629,0.000154,0.018836,0.001371,5.938776,1.034855,30,2.0,2.0,2.0,1.0,2.0,2.0,3.0
Persona_4,189,5.227513,5.666667,10.894180,0.001848,0.000120,0.016582,0.001138,5.619048,0.859471,30,4.0,4.0,4.0,4.0,4.0,4.0,4.0
Persona_1,208,11.379808,10.644231,22.024038,0.002292,0.000404,0.020846,0.001338,10.341346,1.173478,60,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [10]:
rank_summary = rank_df.groupby('persona')[[col + '_rank' for col in metrics]].agg(['mean', 'std'])
rank_summary

mean_in_degree_rank      mean_out_degree_rank           \
                         mean  std                 mean      std   
persona                                                            
Persona_1                 1.0  0.0             1.333333  0.57735   
Persona_2                 3.0  0.0             2.666667  0.57735   
Persona_3                 2.0  0.0             2.000000  1.00000   
Persona_4                 4.0  0.0             4.000000  0.00000   

          mean_total_degree_rank      mean_betweenness_rank           \
                            mean  std                  mean      std   
persona                                                                
Persona_1                    1.0  0.0              2.000000  1.00000   
Persona_2                    3.0  0.0              2.666667  0.57735   
Persona_3                    2.0  0.0              1.333333  0.57735   
Persona_4                    4.0  0.0              4.000000  0.00000   

          mean_eigenvector_rank      mean_pagerank_rank      mean_kcore_rank  \
                           mean  std               mean  std            mean   
persona                                                                        
Persona_1                   1.0  0.0                1.0  0.0        1.000000   
Persona_2                   3.0  0.0                3.0  0.0        2.666667   
Persona_3                   2.0  0.0                2.0  0.0        2.666667   
Persona_4                   4.0  0.0                4.0  0.0        3.666667   

                     
                std  
persona              
Persona_1  0.000000  
Persona_2  1.154701  
Persona_3  0.577350  
Persona_4  0.577350

In [11]:
print(rank_summary)

          mean_in_degree_rank      mean_out_degree_rank           \
                         mean  std                 mean      std   
persona                                                            
Persona_1                 1.0  0.0             1.333333  0.57735   
Persona_2                 3.0  0.0             2.666667  0.57735   
Persona_3                 2.0  0.0             2.000000  1.00000   
Persona_4                 4.0  0.0             4.000000  0.00000   

          mean_total_degree_rank      mean_betweenness_rank           \
                            mean  std                  mean      std   
persona                                                                
Persona_1                    1.0  0.0              2.000000  1.00000   
Persona_2                    3.0  0.0              2.666667  0.57735   
Persona_3                    2.0  0.0              1.333333  0.57735   
Persona_4                    4.0  0.0              4.000000  0.00000   

          mean_eig

The rankings show a very clear and stable hierarchy: Persona_1 consistently dominates the network, holding top positions across all centrality measures (degree, PageRank, eigenvector), indicating both high activity and strong influence. Persona_3 emerges as structurally important despite lower volume, with the best betweenness rank, suggesting it plays a key bridging role in connecting parts of the network. Persona_2 is moderately active but less central, aligning with earlier observations of shorter-lived connections and weaker structural impact. Persona_4 is consistently peripheral, ranking last across all metrics, confirming it as the least influential and most isolated persona.

### Posts

In [13]:
posts = pd.read_sql("SELECT * FROM post", conn3)
posts.head()

,id,tweet,post_img,user_id,comment_to,thread_id,round,news_id,shared_from,image_id,reaction_count
0,1,Hey friends! Just learned about new economic t...,None,17,-1,1,1,None,-1,None,3
1,2,Just analyzed the NBA playoffs schedule & I'm ...,None,411,-1,2,1,None,-1,None,0
2,3,"– @MelissaGardner So do, I saw a sweet post an...",None,411,-1,3,1,None,-1,None,2
3,4,@MelissaGardner I completely agree & what stru...,None,245,2,2,1,None,-1,None,7
4,5,Agree on the value of equality and justice; as...,None,245,1,1,1,None,-1,None,0


In [18]:
len(posts)

140111

In [17]:
personas.rename(columns={'follower_id': 'user_id'}, inplace=True)
posts = posts.merge(personas, on='user_id', how='left')
posts.head()

,id,tweet,post_img,user_id,comment_to,thread_id,round,news_id,shared_from,image_id,reaction_count,persona
0,1,Hey friends! Just learned about new economic t...,None,17,-1,1,1,None,-1,None,3,Persona_2
1,2,Just analyzed the NBA playoffs schedule & I'm ...,None,411,-1,2,1,None,-1,None,0,Persona_4
2,3,"– @MelissaGardner So do, I saw a sweet post an...",None,411,-1,3,1,None,-1,None,2,Persona_4
3,4,@MelissaGardner I completely agree & what stru...,None,245,2,2,1,None,-1,None,7,Persona_4
4,5,Agree on the value of equality and justice; as...,None,245,1,1,1,None,-1,None,0,Persona_4


In [40]:
days = [14, 30, 60]
post_counts = []

for day in days:
    posts_t = posts[posts['round'] <= day * 24]

    counts = posts_t.groupby('persona')['persona'].count().reset_index(name='count')
    counts = pd.DataFrame(counts)
    counts['ratio'] = counts['count'] / len(posts_t)
    counts['day'] = day
    post_counts.append(counts)

post_counts = pd.concat(post_counts)
post_counts

,persona,count,ratio,day
0,Persona_1,8394,0.265877,14
1,Persona_2,7956,0.252003,14
2,Persona_3,8000,0.253397,14
3,Persona_4,7221,0.228723,14
0,Persona_1,18334,0.265860,30
1,Persona_2,17373,0.251925,30
2,Persona_3,17813,0.258305,30
3,Persona_4,15441,0.223909,30
0,Persona_1,38111,0.272006,60
1,Persona_2,34226,0.244278,60


In [41]:
post_summary = post_counts[['persona', 'ratio']].groupby('persona').agg(
    mean_ratio=('ratio', 'mean'),
    std_ratio=('ratio', 'std')
)
post_summary

,mean_ratio,std_ratio
persona,,
Persona_1,0.267914,0.003543
Persona_2,0.249402,0.004438
Persona_3,0.255919,0.002457
Persona_4,0.226765,0.002529


Posting activity is very stable over time, with Persona_1 consistently leading (~26.8%), followed by Persona_3 and Persona_2, while Persona_4 remains the least active. The very low standard deviations indicate that these proportions hardly change, suggesting a persistent and balanced activity structure rather than shifting dominance.

### Reactions

In [46]:
reactions = pd.read_sql("SELECT * FROM reactions", conn3)
reactions = reactions.merge(personas, on='user_id', how='left')
reactions.head()

,id,post_id,user_id,type,round,persona
0,1,1,411,dislike,1,Persona_4
1,2,1,411,dislike,1,Persona_4
2,3,1,411,dislike,1,Persona_4
3,4,4,496,dislike,1,Persona_3
4,5,4,496,dislike,1,Persona_3


In [47]:
len(reactions)

41984

In [48]:
reactions['type'].value_counts()

type
like       21766
dislike    20218
Name: count, dtype: int64

In [56]:
days = [14, 30, 60]
reactions_counts = []

for day in days:
    reactions_t = reactions[reactions['round'] <= day * 24]
    counts = reactions_t.groupby('persona').agg(
        likes=('type', lambda x: (x == 'like').sum()),
        dislikes=('type', lambda x: (x == 'dislike').sum()),
        total=('type', 'count')
    ).reset_index()
    counts['likes_ratio'] = counts['likes'] / len(reactions_t[reactions_t['type'] == 'like'])
    counts['dislikes_ratio'] = counts['dislikes'] / len(reactions_t[reactions_t['type'] == 'dislike'])
    counts['total_ratio'] = counts['total'] / len(reactions_t)
    counts['day'] = day
    reactions_counts.append(counts)

reactions_counts = pd.concat(reactions_counts)
reactions_counts

,persona,likes,dislikes,total,likes_ratio,dislikes_ratio,total_ratio,day
0,Persona_1,1321,1145,2466,0.262311,0.242174,0.252560,14
1,Persona_2,1243,1209,2452,0.246823,0.255711,0.251127,14
2,Persona_3,1497,1164,2661,0.297260,0.246193,0.272532,14
3,Persona_4,975,1210,2185,0.193606,0.255922,0.223781,14
0,Persona_1,2925,2421,5346,0.270934,0.241400,0.256711,30
1,Persona_2,2638,2568,5206,0.244350,0.256057,0.249988,30
2,Persona_3,3250,2448,5698,0.301037,0.244092,0.273613,30
3,Persona_4,1983,2592,4575,0.183679,0.258450,0.219688,30
0,Persona_1,6105,5160,11265,0.280483,0.255218,0.268317,60
1,Persona_2,5300,5020,10320,0.243499,0.248294,0.245808,60


In [60]:
reactions_summary = (reactions_counts.groupby('persona')[['likes_ratio', 'dislikes_ratio', 'total_ratio']].
                     agg(['mean', 'std']))
print(reactions_summary)

          likes_ratio           dislikes_ratio           total_ratio          
                 mean       std           mean       std        mean       std
persona                                                                       
Persona_1    0.271243  0.009090       0.246264  0.007764    0.259196  0.008167
Persona_2    0.244891  0.001727       0.253354  0.004386    0.248974  0.002801
Persona_3    0.297873  0.002906       0.242846  0.004114    0.271328  0.003069
Persona_4    0.185993  0.006760       0.257536  0.001402    0.220502  0.002958


This reveals a clear asymmetry between activity and reception of feedback across personas. Persona_3 stands out as the most positively received, with the highest like ratio and relatively low dislike share, suggesting its content resonates best with others. In contrast, Persona_4 is the least favored, having the lowest like ratio and the highest dislike ratio, indicating weaker alignment with the audience. Persona_1 maintains strong overall engagement (high total ratio) with balanced sentiment, while Persona_2 appears neutral, receiving a fairly even split of likes and dislikes without strong preference.

### Sentiment

In [61]:
posts = posts[['persona', 'tweet', 'round']]
posts.head()

,persona,tweet,round
0,Persona_2,Hey friends! Just learned about new economic t...,1
1,Persona_4,Just analyzed the NBA playoffs schedule & I'm ...,1
2,Persona_4,"– @MelissaGardner So do, I saw a sweet post an...",1
3,Persona_4,@MelissaGardner I completely agree & what stru...,1
4,Persona_4,Agree on the value of equality and justice; as...,1


In [63]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

posts['polarity'] = posts['tweet'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

posts.head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\magda\AppData\Roaming\nltk_data...
C:\Users\magda\AppData\Local\Temp\ipykernel_14044\116181813.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  posts['polarity'] = posts['tweet'].apply(


,persona,tweet,round,polarity
0,Persona_2,Hey friends! Just learned about new economic t...,1,0.8591
1,Persona_4,Just analyzed the NBA playoffs schedule & I'm ...,1,0.5707
2,Persona_4,"– @MelissaGardner So do, I saw a sweet post an...",1,0.4144
3,Persona_4,@MelissaGardner I completely agree & what stru...,1,0.4641
4,Persona_4,Agree on the value of equality and justice; as...,1,0.8338


In [69]:
conditions = [
    posts['polarity'] > 0.05,
    posts['polarity'].between(-0.05, 0.05)
]

choices = ['Positive', 'Neutral']

posts['polarity_category'] = np.select(
    conditions,
    choices,
    default='Negative'
)

posts['polarity_category'].value_counts()

C:\Users\magda\AppData\Local\Temp\ipykernel_14044\1304829021.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  posts['polarity_category'] = np.select(


polarity_category
Positive    128041
Negative      6914
Neutral       5156
Name: count, dtype: int64

Most of the agents' posts is positive.

In [75]:
sentiment_summary = posts.groupby('persona').agg(
    positive=('polarity_category', lambda x: (x == 'Positive').sum()),
    neutral=('polarity_category', lambda x: (x == 'Neutral').sum()),
    negative=('polarity_category', lambda x: (x == 'Negative').sum()),
    total=('polarity_category', 'count')
).reset_index()

# sentiment_summary['positive_ratio'] = sentiment_summary['positive'] / sentiment_summary['total']
# sentiment_summary['neutral_ratio'] = sentiment_summary['neutral'] / sentiment_summary['total']
# sentiment_summary['negative_ratio'] = sentiment_summary['negative'] / sentiment_summary['total']

sentiment_summary['positive_ratio'] = sentiment_summary['positive'] / len(posts[posts['polarity_category'] == 'Positive'])
sentiment_summary['neutral_ratio'] = sentiment_summary['neutral'] / len(posts[posts['polarity_category'] == 'Neutral'])
sentiment_summary['negative_ratio'] = sentiment_summary['negative'] / len(posts[posts['polarity_category'] == 'Negative'])

sentiment_summary['total_ratio'] = sentiment_summary['total'] / len(posts)

sentiment_summary

,persona,positive,neutral,negative,total,positive_ratio,neutral_ratio,negative_ratio,total_ratio
0,Persona_1,34936,1401,1774,38111,0.272850,0.271722,0.256581,0.272006
1,Persona_2,31416,1187,1623,34226,0.245359,0.230217,0.234741,0.244278
2,Persona_3,32923,1329,1624,35876,0.257129,0.257758,0.234886,0.256054
3,Persona_4,28766,1239,1893,31898,0.224662,0.240303,0.273792,0.227662


In [77]:
days = [14, 30, 60]
sentiment_statistics = []

for day in days:
    posts_t = posts[posts['round'] <= day * 24]
    sentiment = posts_t.groupby('persona').agg(
        positive=('polarity_category', lambda x: (x == 'Positive').sum()),
        neutral=('polarity_category', lambda x: (x == 'Neutral').sum()),
        negative=('polarity_category', lambda x: (x == 'Negative').sum()),
        total=('polarity_category', 'count')
    ).reset_index()

    sentiment['positive_ratio'] = sentiment['positive'] / len(posts_t[posts_t['polarity_category'] == 'Positive'])
    sentiment['neutral_ratio'] = sentiment['neutral'] / len(posts_t[posts_t['polarity_category'] == 'Neutral'])
    sentiment['negative_ratio'] = sentiment['negative'] / len(posts_t[posts_t['polarity_category'] == 'Negative'])

    sentiment['total_ratio'] = sentiment['total'] / len(posts_t)
    sentiment['day'] = day
    sentiment_statistics.append(sentiment)

sentiment_statistics = pd.concat(sentiment_statistics)
sentiment_statistics

,persona,positive,neutral,negative,total,positive_ratio,neutral_ratio,negative_ratio,total_ratio,day
0,Persona_1,7693,280,421,8394,0.266461,0.253623,0.263784,0.265877,14
1,Persona_2,7273,254,429,7956,0.251914,0.230072,0.268797,0.252003,14
2,Persona_3,7308,314,378,8000,0.253126,0.284420,0.236842,0.253397,14
3,Persona_4,6597,256,368,7221,0.228499,0.231884,0.230576,0.228723,14
0,Persona_1,16813,640,881,18334,0.266962,0.250882,0.256776,0.265860,30
1,Persona_2,15944,616,813,17373,0.253164,0.241474,0.236957,0.251925,30
2,Persona_3,16309,668,836,17813,0.258959,0.261858,0.243661,0.258305,30
3,Persona_4,13913,627,901,15441,0.220915,0.245786,0.262606,0.223909,30
0,Persona_1,34936,1401,1774,38111,0.272850,0.271722,0.256581,0.272006,60
1,Persona_2,31416,1187,1623,34226,0.245359,0.230217,0.234741,0.244278,60


In [79]:
sentiment_statistics_sum = sentiment_statistics.groupby('persona')[['positive_ratio', 'neutral_ratio', 'negative_ratio', 'total_ratio']].agg(['mean', 'std'])
print(sentiment_statistics_sum)

          positive_ratio           neutral_ratio           negative_ratio  \
                    mean       std          mean       std           mean   
persona                                                                     
Persona_1       0.268758  0.003553      0.258742  0.011324       0.259047   
Persona_2       0.250145  0.004192      0.233921  0.006541       0.246832   
Persona_3       0.256405  0.002983      0.268012  0.014357       0.238463   
Persona_4       0.224692  0.003792      0.239324  0.007002       0.255658   

                    total_ratio            
                std        mean       std  
persona                                    
Persona_1  0.004104    0.267914  0.003543  
Persona_2  0.019055    0.249402  0.004438  
Persona_3  0.004607    0.255919  0.002457  
Persona_4  0.022430    0.226765  0.002529  


These results are similar to reactions analysis.

## Behaviour consistency

In [81]:
window = 7 # 7 days window

### Follow

In [94]:
days = [14, 30, 60]
global_summary = []
summaries = []

for day in days:
    follow_t = follow[(follow['round'] <= day * 24) & (follow['round'] > (day - window) * 24)]
    print(len(follow_t))

    global_metrics, summary = build_graph_and_analyse(follow_t, persona_dict)
    global_metrics['day'] = day
    summary['day'] = day

    global_summary.append(global_metrics)
    summaries.append(summary)

2094
Graph creation ...

Number of nodes: 468
Number of edges: 1382
Number of connective components: 35
Components sizes: [405, 1, 3, 1, 2, 2, 4, 1, 1, 1, 2, 2, 1, 2, 2, 2, 1, 1, 2, 2, 5, 3, 1, 2, 2, 4, 1, 2, 2, 1, 2, 2, 1, 1, 1]
Number of nodes (LCC): 405
Number of edges (LCC): 1353

Global metrics ...

Mean degree: 5.91
Density: 0.0063
Diameter: 9
Avg. shortest path: 3.468
Modularity score: 0.387
Persona assortativity: -0.018439353679628297

Local metrics ...


Statistical check (metrics vs. persona) ...

in_degree statistics: p = 0.7769
out_degree statistics: p = 0.8014
total_degree statistics: p = 0.8149
betweenness statistics: p = 0.8974
eigenvector statistics: p = 0.8520
pagerank statistics: p = 0.8386
kcore statistics: p = 0.8087
1944
Graph creation ...

Number of nodes: 475
Number of edges: 1274
Number of connective components: 24
Components sizes: [435, 1, 2, 2, 1, 2, 3, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 1, 2, 2, 3, 2, 1, 1]
Number of nodes (LCC): 435
Number of edges (LCC): 1257



7 days = around 2000 actions

In [84]:
summaries_df = pd.concat(summaries)
summaries_df

,n_nodes,mean_in_degree,mean_out_degree,mean_total_degree,mean_betweenness,median_betweenness,mean_eigenvector,mean_pagerank,mean_kcore,pagerank_ratio,day
persona,,,,,,,,,,,
Persona_1,101,3.594059,3.445545,7.039604,0.004393,0.000477,0.030601,0.002796,3.603960,1.132271,14
Persona_2,110,2.863636,3.227273,6.090909,0.004076,0.000311,0.026935,0.002146,3.172727,0.869307,14
Persona_3,100,3.700000,3.400000,7.100000,0.004494,0.000164,0.030541,0.002640,3.450000,1.069258,14
Persona_4,94,3.244681,3.297872,6.542553,0.003318,0.000550,0.027494,0.002314,3.489362,0.937138,14
Persona_1,105,3.523810,3.266667,6.790476,0.004083,0.000416,0.032823,0.002591,3.333333,1.127148,30
Persona_2,117,2.504274,2.752137,5.256410,0.003164,0.000238,0.025216,0.002109,2.923077,0.917414,30
Persona_3,113,3.238938,2.920354,6.159292,0.004246,0.000142,0.028455,0.002493,2.823009,1.084508,30
Persona_4,100,2.280000,2.620000,4.900000,0.002727,0.000129,0.024135,0.001995,2.900000,0.867625,30
Persona_1,113,2.973451,2.884956,5.858407,0.004435,0.000644,0.033039,0.002486,3.053097,1.098696,60


In [85]:
global_summary_df = pd.concat(global_summary)
global_summary_df

,Simulation,day
Metric,,
Mean degree,5.905983,14
Density,0.006323,14
Diameter,9.000000,14
Avg. shortest path,3.468354,14
Modularity,0.386964,14
Persona assortativity,-0.018439,14
Mean degree,5.364211,30
Density,0.005658,30
Diameter,10.000000,30


In [86]:
rank_df = summaries_df.copy()

metrics = ['mean_in_degree', 'mean_out_degree', 'mean_total_degree', 'mean_betweenness', 'mean_eigenvector', 'mean_pagerank', 'mean_kcore']

for col in metrics:
    rank_df[col + '_rank'] = rank_df.groupby('day')[col] \
                                   .rank(ascending=False, method='min')

rank_df

,n_nodes,mean_in_degree,mean_out_degree,mean_total_degree,mean_betweenness,median_betweenness,mean_eigenvector,mean_pagerank,mean_kcore,pagerank_ratio,day,mean_in_degree_rank,mean_out_degree_rank,mean_total_degree_rank,mean_betweenness_rank,mean_eigenvector_rank,mean_pagerank_rank,mean_kcore_rank
persona,,,,,,,,,,,,,,,,,,
Persona_1,101,3.594059,3.445545,7.039604,0.004393,0.000477,0.030601,0.002796,3.603960,1.132271,14,2.0,1.0,2.0,2.0,1.0,1.0,1.0
Persona_2,110,2.863636,3.227273,6.090909,0.004076,0.000311,0.026935,0.002146,3.172727,0.869307,14,4.0,4.0,4.0,3.0,4.0,4.0,4.0
Persona_3,100,3.700000,3.400000,7.100000,0.004494,0.000164,0.030541,0.002640,3.450000,1.069258,14,1.0,2.0,1.0,1.0,2.0,2.0,3.0
Persona_4,94,3.244681,3.297872,6.542553,0.003318,0.000550,0.027494,0.002314,3.489362,0.937138,14,3.0,3.0,3.0,4.0,3.0,3.0,2.0
Persona_1,105,3.523810,3.266667,6.790476,0.004083,0.000416,0.032823,0.002591,3.333333,1.127148,30,1.0,1.0,1.0,2.0,1.0,1.0,1.0
Persona_2,117,2.504274,2.752137,5.256410,0.003164,0.000238,0.025216,0.002109,2.923077,0.917414,30,3.0,3.0,3.0,3.0,3.0,3.0,2.0
Persona_3,113,3.238938,2.920354,6.159292,0.004246,0.000142,0.028455,0.002493,2.823009,1.084508,30,2.0,2.0,2.0,1.0,2.0,2.0,4.0
Persona_4,100,2.280000,2.620000,4.900000,0.002727,0.000129,0.024135,0.001995,2.900000,0.867625,30,4.0,4.0,4.0,4.0,4.0,4.0,3.0
Persona_1,113,2.973451,2.884956,5.858407,0.004435,0.000644,0.033039,0.002486,3.053097,1.098696,60,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [87]:
rank_summary = rank_df.groupby('persona')[[col + '_rank' for col in metrics]].agg(['mean', 'std'])
print(rank_summary)

          mean_in_degree_rank          mean_out_degree_rank           \
                         mean      std                 mean      std   
persona                                                                
Persona_1            1.333333  0.57735             1.000000  0.00000   
Persona_2            3.666667  0.57735             3.666667  0.57735   
Persona_3            1.666667  0.57735             2.000000  0.00000   
Persona_4            3.333333  0.57735             3.333333  0.57735   

          mean_total_degree_rank          mean_betweenness_rank           \
                            mean      std                  mean      std   
persona                                                                    
Persona_1               1.333333  0.57735              1.666667  0.57735   
Persona_2               3.666667  0.57735              3.333333  0.57735   
Persona_3               1.666667  0.57735              1.333333  0.57735   
Persona_4               3.333333  0.577

The ranking is still very consistent, but now you can see more nuance and variability across time windows. Persona_1 clearly remains the central leader, dominating eigenvector, PageRank, and k-core with zero variance, indicating a stable core position. Persona_3 emerges as a strong secondary actor, especially with the best betweenness rank, reinforcing its role as a bridge in the network. Meanwhile, Persona_2 and Persona_4 stay peripheral, but Persona_2 shows slightly higher variability, suggesting more unstable or context-dependent behavior.

### Posts

In [95]:
days = [14, 30, 60]
post_counts = []

for day in days:
    posts_t = posts[(posts['round'] <= day * 24) & (posts['round'] > (day - window) * 24)]
    print(len(posts_t))

    counts = posts_t.groupby('persona')['persona'].count().reset_index(name='count')
    counts = pd.DataFrame(counts)
    counts['ratio'] = counts['count'] / len(posts_t)
    counts['day'] = day
    post_counts.append(counts)

post_counts = pd.concat(post_counts)
post_counts

16474
16580
16881


,persona,count,ratio,day
0,Persona_1,4360,0.264659,14
1,Persona_2,4063,0.246631,14
2,Persona_3,4217,0.255979,14
3,Persona_4,3834,0.232730,14
0,Persona_1,4541,0.273884,30
1,Persona_2,4052,0.244391,30
2,Persona_3,4481,0.270265,30
3,Persona_4,3506,0.211460,30
0,Persona_1,4719,0.279545,60
1,Persona_2,3921,0.232273,60


In [89]:
post_summary = post_counts[['persona', 'ratio']].groupby('persona').agg(
    mean_ratio=('ratio', 'mean'),
    std_ratio=('ratio', 'std')
)
post_summary

,mean_ratio,std_ratio
persona,,
Persona_1,0.272696,0.007514
Persona_2,0.241098,0.007725
Persona_3,0.256946,0.012863
Persona_4,0.229259,0.016343


The posting distribution remains fairly stable, with Persona_1 still leading, followed by Persona_3, while Persona_4 is the least active. However, compared to earlier results, variability has increased (higher std), especially for Persona_3 and Persona_4, indicating more fluctuation in their activity across time windows. This suggests that while the overall hierarchy persists, short-term dynamics are more pronounced, particularly for less dominant personas.

### Reactions

In [96]:
days = [14, 30, 60]
reactions_counts = []

for day in days:
    reactions_t = reactions[(reactions['round'] <= day * 24) & (reactions['round'] > (day - window) * 24)]
    print(len(reactions_t))

    counts = reactions_t.groupby('persona').agg(
        likes=('type', lambda x: (x == 'like').sum()),
        dislikes=('type', lambda x: (x == 'dislike').sum()),
        total=('type', 'count')
    ).reset_index()
    counts['likes_ratio'] = counts['likes'] / len(reactions_t[reactions_t['type'] == 'like'])
    counts['dislikes_ratio'] = counts['dislikes'] / len(reactions_t[reactions_t['type'] == 'dislike'])
    counts['total_ratio'] = counts['total'] / len(reactions_t)
    counts['day'] = day
    reactions_counts.append(counts)

reactions_counts = pd.concat(reactions_counts)
reactions_counts

4930
4852
4962


,persona,likes,dislikes,total,likes_ratio,dislikes_ratio,total_ratio,day
0,Persona_1,698,573,1271,0.276326,0.238353,0.257809,14
1,Persona_2,614,598,1212,0.243072,0.248752,0.245842,14
2,Persona_3,737,614,1351,0.291766,0.255408,0.274037,14
3,Persona_4,477,619,1096,0.188836,0.257488,0.222312,14
0,Persona_1,773,568,1341,0.292250,0.257363,0.276381,30
1,Persona_2,608,533,1141,0.229868,0.241504,0.235161,30
2,Persona_3,834,536,1370,0.315312,0.242864,0.282358,30
3,Persona_4,430,570,1000,0.162571,0.258269,0.206101,30
0,Persona_1,736,612,1348,0.286047,0.256174,0.271665,60
1,Persona_2,630,585,1215,0.244850,0.244872,0.244861,60


In [91]:
reactions_summary = (reactions_counts.groupby('persona')[['likes_ratio', 'dislikes_ratio', 'total_ratio']].
                     agg(['mean', 'std']))
print(reactions_summary)

          likes_ratio           dislikes_ratio           total_ratio          
                 mean       std           mean       std        mean       std
persona                                                                       
Persona_1    0.284874  0.008026       0.250630  0.010649    0.268618  0.009653
Persona_2    0.239263  0.008185       0.245043  0.003627    0.241954  0.005904
Persona_3    0.298745  0.014407       0.241994  0.013869    0.271989  0.011530
Persona_4    0.177118  0.013359       0.262333  0.007726    0.217438  0.009851


Persona_3 stands out as the most positively reactive, giving the highest share of likes and relatively fewer dislikes, suggesting a more supportive or agreeable behavior. Persona_4 is clearly the most critical, with the lowest like ratio and highest dislike ratio, indicating a tendency to evaluate others more negatively. Persona_1 is moderately positive and active, while Persona_2 appears the most neutral, with nearly balanced like/dislike behavior.

Combined with your previous results, this suggests:

Persona_3 = positive both in content and reactions
Persona_4 = negative both in content and reactions
Persona_1 = influential but balanced
Persona_2 = relatively indifferent / non-committal

### Sentiment

In [92]:
days = [14, 30, 60]
sentiment_statistics = []

for day in days:
    posts_t = posts[(posts['round'] <= day * 24) & (posts['round'] > (day - window) * 24)]
    sentiment = posts_t.groupby('persona').agg(
        positive=('polarity_category', lambda x: (x == 'Positive').sum()),
        neutral=('polarity_category', lambda x: (x == 'Neutral').sum()),
        negative=('polarity_category', lambda x: (x == 'Negative').sum()),
        total=('polarity_category', 'count')
    ).reset_index()

    sentiment['positive_ratio'] = sentiment['positive'] / len(posts_t[posts_t['polarity_category'] == 'Positive'])
    sentiment['neutral_ratio'] = sentiment['neutral'] / len(posts_t[posts_t['polarity_category'] == 'Neutral'])
    sentiment['negative_ratio'] = sentiment['negative'] / len(posts_t[posts_t['polarity_category'] == 'Negative'])

    sentiment['total_ratio'] = sentiment['total'] / len(posts_t)
    sentiment['day'] = day
    sentiment_statistics.append(sentiment)

sentiment_statistics = pd.concat(sentiment_statistics)
sentiment_statistics

,persona,positive,neutral,negative,total,positive_ratio,neutral_ratio,negative_ratio,total_ratio,day
0,Persona_1,3994,143,223,4360,0.265364,0.246978,0.264218,0.264659,14
1,Persona_2,3731,131,201,4063,0.247891,0.226252,0.238152,0.246631,14
2,Persona_3,3831,172,214,4217,0.254535,0.297064,0.253555,0.255979,14
3,Persona_4,3495,133,206,3834,0.232210,0.229706,0.244076,0.232730,14
0,Persona_1,4203,147,191,4541,0.276987,0.234824,0.244872,0.273884,30
1,Persona_2,3717,151,184,4052,0.244958,0.241214,0.235897,0.244391,30
2,Persona_3,4128,160,193,4481,0.272044,0.255591,0.247436,0.270265,30
3,Persona_4,3126,168,212,3506,0.206010,0.268371,0.271795,0.211460,30
0,Persona_1,4348,180,191,4719,0.280498,0.287081,0.253652,0.279545,60
1,Persona_2,3610,128,183,3921,0.232888,0.204147,0.243028,0.232273,60


In [93]:
sentiment_statistics_sum = sentiment_statistics.groupby('persona')[['positive_ratio', 'neutral_ratio', 'negative_ratio', 'total_ratio']].agg(['mean', 'std'])
print(sentiment_statistics_sum)

          positive_ratio           neutral_ratio           negative_ratio  \
                    mean       std          mean       std           mean   
persona                                                                     
Persona_1       0.274283  0.007921      0.256294  0.027346       0.254247   
Persona_2       0.241912  0.007951      0.223871  0.018648       0.239026   
Persona_3       0.256747  0.014320      0.270343  0.023183       0.251105   
Persona_4       0.227058  0.019002      0.249492  0.019348       0.255622   

                    total_ratio            
                std        mean       std  
persona                                    
Persona_1  0.009687    0.272696  0.007514  
Persona_2  0.003645    0.241098  0.007725  
Persona_3  0.003236    0.256946  0.012863  
Persona_4  0.014427    0.229259  0.016343  


The sentiment patterns remain broadly consistent, with Persona_1 still the most positive overall, while Persona_4 shows the highest negativity and lowest positivity, reinforcing its more critical tone. Persona_3 continues to lean neutral, suggesting a balanced, less polarizing communication style, whereas Persona_2 appears slightly subdued, with lower positive and overall sentiment intensity. Increased standard deviations—especially for Persona_4—indicate that less dominant personas exhibit more variability in tone over time, while leading personas remain more consistent.

## Conclusion

- Personas behave consistently not only over time, but also across behavioral dimensions (posting, reacting, sentiment, network position)
- agents exhibit stable, persona-specific behavioral patterns over time, with consistent differences in activity, sentiment, network position, and interaction style, indicating alignment between assigned personalities and observed behavior.